# Notebook 04: Full RAG Pipeline

This is the capstone notebook. We'll combine everything from notebooks 01-03 into a complete, working RAG system.

## The Complete RAG Pipeline

```
INDEXING PHASE (done once)
─────────────────────────
Raw Documents
    │
    ▼
Chunk Documents  ──▶  Embed Chunks  ──▶  Store in Vector DB


QUERY PHASE (done on every user question)
─────────────────────────────────────────
User Question
    │
    ▼
Embed Question  ──▶  Search Vector DB  ──▶  Top-K Chunks
                                                  │
                                                  ▼
                              Build Prompt: [System] + [Context] + [Question]
                                                  │
                                                  ▼
                                           LLM Generation
                                                  │
                                                  ▼
                                            Final Answer
```

## What you'll build:
1. Index the knowledge base (chunk → embed → store)
2. Build a retrieval function
3. Build a generation function using the OpenAI API
4. Combine into a full RAG chain
5. **Compare RAG vs no-RAG** to see the difference

**Requires:** An OpenAI API key in your `.env` file.

## Setup

In [ ]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI

# Load API key from .env file
load_dotenv("../.env")
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Copy .env.example to .env and add your key.")

client_openai = OpenAI(api_key=api_key)
print("OpenAI client ready")

## Phase 1: Indexing — Chunk, Embed, and Store

In [ ]:
def load_and_chunk_knowledge_base(file_path: str) -> list[dict]:
    """Load the knowledge base and split at DOCUMENT: boundaries."""
    text = Path(file_path).read_text()
    sections = text.strip().split('\n\n')
    chunks = []
    for section in sections:
        section = section.strip()
        if section.startswith('DOCUMENT:'):
            lines = section.split('\n', 1)
            title = lines[0].replace('DOCUMENT:', '').strip()
            content = lines[1].strip() if len(lines) > 1 else ""
            if content:
                chunks.append({"title": title, "content": content})
    return chunks


# Load and chunk
chunks = load_and_chunk_knowledge_base("../data/knowledge_base.txt")
print(f"Loaded {len(chunks)} document chunks")
for chunk in chunks:
    print(f"  - {chunk['title']} ({len(chunk['content'])} chars)")

In [ ]:
# Set up ChromaDB with sentence-transformers embeddings
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

chroma_client = chromadb.PersistentClient(path="../data/chroma_rag_final")

try:
    chroma_client.delete_collection("rag_kb")
except:
    pass

collection = chroma_client.create_collection(
    name="rag_kb",
    embedding_function=embedding_fn
)

# Index all chunks
collection.add(
    documents=[c["content"] for c in chunks],
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    metadatas=[{"title": c["title"]} for c in chunks]
)

print(f"Indexed {collection.count()} chunks into ChromaDB")

## Phase 2: Retrieval Function

Given a user query, find the most relevant chunks from the vector store.

In [ ]:
def retrieve(query: str, n_results: int = 3) -> list[dict]:
    """
    Retrieve the most relevant document chunks for a query.
    
    Returns a list of {title, content, score} dicts.
    """
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    
    retrieved = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        retrieved.append({
            "title": meta["title"],
            "content": doc,
            "score": round(1 / (1 + dist), 3)
        })
    
    return retrieved


# Test retrieval
test_query = "How does RAG reduce hallucinations?"
retrieved_docs = retrieve(test_query)

print(f"Query: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} chunks:\n")
for doc in retrieved_docs:
    print(f"  [{doc['score']}] {doc['title']}")
    print(f"    {doc['content'][:120]}...\n")

## Phase 3: Generation Functions

Two versions: one that uses RAG context, one that doesn't. This lets us compare the difference.

In [ ]:
def generate_without_rag(question: str) -> str:
    """Ask the LLM directly without any retrieved context."""
    response = client_openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. Answer the user's question."
            },
            {
                "role": "user",
                "content": question
            }
        ],
        temperature=0.0,
        max_tokens=400
    )
    return response.choices[0].message.content


def generate_with_rag(question: str, n_chunks: int = 3) -> dict:
    """
    Full RAG pipeline: retrieve context, then generate a grounded answer.
    
    Returns the answer and the retrieved context for inspection.
    """
    # Step 1: Retrieve relevant chunks
    context_docs = retrieve(question, n_results=n_chunks)
    
    # Step 2: Format context for the prompt
    context_text = "\n\n".join([
        f"[Source: {doc['title']}]\n{doc['content']}"
        for doc in context_docs
    ])
    
    # Step 3: Build the augmented prompt
    system_prompt = """You are a helpful assistant. Answer the user's question using ONLY 
the provided context. If the context doesn't contain enough information to answer, 
say so explicitly. Do not make up information."""
    
    user_prompt = f"""Context:
---
{context_text}
---

Question: {question}"""
    
    # Step 4: Generate
    response = client_openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0,
        max_tokens=500
    )
    
    return {
        "answer": response.choices[0].message.content,
        "context": context_docs,
        "prompt_preview": user_prompt[:400] + "..."
    }

## Phase 4: Compare RAG vs No-RAG

This is the key demonstration. We'll ask questions that require specific knowledge from our documents.

In [ ]:
def compare_rag_vs_no_rag(question: str):
    """Run a question through both pipelines and print a comparison."""
    print("=" * 70)
    print(f"QUESTION: {question}")
    print("=" * 70)
    
    # Without RAG
    print("\n[WITHOUT RAG] Direct LLM answer:")
    print("-" * 40)
    answer_no_rag = generate_without_rag(question)
    print(answer_no_rag)
    
    # With RAG
    print("\n[WITH RAG] Retrieved context + grounded answer:")
    print("-" * 40)
    result = generate_with_rag(question)
    
    print("Retrieved sources:")
    for doc in result['context']:
        print(f"  - [{doc['score']}] {doc['title']}")
    
    print("\nAnswer:")
    print(result['answer'])
    print()

In [ ]:
# Question 1: Specific factual question
compare_rag_vs_no_rag(
    "When was RAG introduced and by which organization?"
)

In [ ]:
# Question 2: Conceptual question requiring our specific framing
compare_rag_vs_no_rag(
    "What are the four main stages of a RAG pipeline?"
)

In [ ]:
# Question 3: Comparison question
compare_rag_vs_no_rag(
    "When should I use fine-tuning versus RAG for my LLM application?"
)

## Phase 5: Interactive RAG Chat

A simple interactive loop to ask your own questions.

In [ ]:
def rag_chat():
    """Simple interactive RAG Q&A loop."""
    print("RAG Q&A System — ask anything about AI/ML concepts")
    print("Type 'quit' to exit\n")
    
    while True:
        question = input("Your question: ").strip()
        if question.lower() in ('quit', 'exit', 'q'):
            print("Goodbye!")
            break
        if not question:
            continue
        
        result = generate_with_rag(question)
        
        print(f"\nSources used:")
        for doc in result['context']:
            print(f"  - {doc['title']} (score: {doc['score']})")
        
        print(f"\nAnswer: {result['answer']}\n")
        print("-" * 60)


# Uncomment to run interactively
# rag_chat()

## Inspecting the RAG Prompt

Let's look at exactly what gets sent to the LLM — this is key for debugging and tuning.

In [ ]:
question = "What chunking strategies work best for RAG?"
context_docs = retrieve(question, n_results=2)

context_text = "\n\n".join([
    f"[Source: {doc['title']}]\n{doc['content']}"
    for doc in context_docs
])

full_prompt = f"""SYSTEM: You are a helpful assistant. Answer using ONLY the provided context.

USER:
Context:
---
{context_text}
---

Question: {question}"""

print("=== FULL PROMPT SENT TO LLM ===")
print(full_prompt)
print(f"\nTotal prompt length: {len(full_prompt)} characters")

## Key Takeaways

### The RAG Pipeline:
```
1. LOAD     → Read your documents
2. CHUNK    → Split into retrievable pieces 
3. EMBED    → Convert chunks to vectors
4. STORE    → Index vectors in a vector database
5. RETRIEVE → Find relevant chunks for each query
6. AUGMENT  → Inject chunks into the LLM prompt
7. GENERATE → LLM produces a grounded answer
```

### RAG vs No-RAG:
| | Without RAG | With RAG |
|---|---|---|
| Knowledge source | Training data only | Your documents |
| Hallucinations | More likely | Reduced |
| Up-to-date info | No | Yes |
| Citeable sources | No | Yes |
| Cost | Lower | Slightly higher |

### What to tune:
- **Chunk size** — affects retrieval precision and context richness
- **Top-K** — how many chunks to retrieve (2-5 is typical)
- **Embedding model** — better models = better retrieval
- **System prompt** — controls how the LLM uses the context

---

For a clean, reusable implementation, see `src/rag_pipeline.py`.